In [ ]:
!pip install scikit-posthocs xgboost

In [ ]:
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────
# CONCEPT — SLIDING WINDOW:
# 
# We convert the time series into supervised learning.
# 
# For each position i in the series:
#   INPUT (X):  glucose[i], glucose[i+1], ..., glucose[i+11]
#               = last 12 readings = last 60 minutes
#   
#   TARGET (y): glucose[i+12+5]
#               = glucose 6 steps after the window ends
#               = 30 minutes into the future
#
# Example:
#   Window: [120, 122, 125, 128, 130, 132, 135, 138, 140, 142, 144, 146]
#   Target: 155  (what glucose will be 30 min later)
#
# We slide this window one step at a time across the 
# entire time series to create thousands of training examples.
# ─────────────────────────────────────────────────

def make_windows(glucose_array, window_size=12, horizon=6):
    """
    Create (X, y) pairs using sliding window.
    
    window_size = 12 steps = 60 minutes of input history
    horizon     = 6 steps  = 30 minutes ahead to predict
    """
    X, y = [], []
    n = len(glucose_array)
    
    for i in range(n - window_size - horizon):
        window = glucose_array[i : i + window_size]
        target = glucose_array[i + window_size + horizon - 1]
        
        # Skip if any NaN remains
        if np.isnan(window).any() or np.isnan(target):
            continue
        
        X.append(window)
        y.append(target)
    
    return np.array(X), np.array(y)


def time_series_split(X, y, test_ratio=0.2):
    """
    CONCEPT — WHY NO SHUFFLE:
    In time series, future data cannot influence past predictions.
    If we shuffle and split randomly, readings from tomorrow 
    end up in training and readings from yesterday in test.
    This causes 'data leakage' — unrealistically good results.
    
    Correct approach: first 80% = train, last 20% = test.
    This mirrors real deployment: train on past, test on future.
    """
    split = int(len(X) * (1 - test_ratio))
    return X[:split], X[split:], y[:split], y[split:]


def prepare_column(df, col_name):
    """Full pipeline for one preprocessed column."""
    arr = df[col_name].values.astype(float)
    X, y = make_windows(arr)
    X_train, X_test, y_train, y_test = time_series_split(X, y)
    return X_train, X_test, y_train, y_test





In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import os

# ─────────────────────────────────────────────────
# CONCEPT — XGBoost:
# Builds an ensemble of decision trees sequentially.
# Each tree corrects the errors of the previous one.
# Takes flat feature vector (12 glucose values) as input.
# Does NOT understand time order — just sees 12 numbers.
# ─────────────────────────────────────────────────

def run_xgboost(df, col_name, seed=42):
    X_train, X_test, y_train, y_test = prepare_column(df, col_name)
    
    model = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        verbosity=0
    )
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)
    
    y_pred = model.predict(X_test)
    rmse = np.sqrt(np.mean((y_test - y_pred)**2))
    return rmse, y_test, y_pred





In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ─────────────────────────────────────────────────
# CONCEPT — 1D-CNN:
# Applies convolutional filters along the TIME axis.
# A filter of size 3 looks at 3 consecutive glucose 
# readings at a time and detects local patterns:
# rising trend, plateau, sharp drop, etc.
# Stacking two conv layers detects progressively 
# more abstract patterns.
# Needs input shape: (samples, 1, timesteps)  ← PyTorch channel-first
#
# NOTE: Uses PyTorch instead of TensorFlow because
# TensorFlow has no wheel for Python 3.14.
# Architecture is identical.
# ─────────────────────────────────────────────────

class CNN1D(nn.Module):
    """
    1D-CNN for glucose forecasting.
    
    Input:  (batch, 1, 12)  — 12 time-steps, 1 channel
    Output: (batch, 1)      — next glucose value
    """
    def __init__(self, seq_len=12):
        super().__init__()
        self.net = nn.Sequential(
            # Conv layer 1: detect local glucose patterns
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),              # seq_len → 6

            # Conv layer 2: detect higher-level patterns
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),                 # 32 × 6 = 192

            nn.Linear(32 * (seq_len // 2), 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def run_cnn(df, col_name, epochs=30, patience=10, batch_size=128, max_samples=50_000, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    X_train, X_test, y_train, y_test = prepare_column(df, col_name)

    # Subsample training data so CPU training completes in reasonable time.
    # 50 000 samples ≈ 69 days of 5-min CGM readings — more than enough signal.
    if len(X_train) > max_samples:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X_train), max_samples, replace=False)
        idx.sort()                      # keep temporal order
        X_train, y_train = X_train[idx], y_train[idx]

    # Scale to [0,1] — neural nets train better on small values
    sx = MinMaxScaler()
    sy = MinMaxScaler()
    X_train_s = sx.fit_transform(X_train)
    X_test_s  = sx.transform(X_test)
    y_train_s = sy.fit_transform(y_train.reshape(-1, 1)).flatten()

    # PyTorch tensors — shape (N, 1, 12) for Conv1d channel-first
    X_tr = torch.tensor(X_train_s, dtype=torch.float32).unsqueeze(1)
    X_te = torch.tensor(X_test_s,  dtype=torch.float32).unsqueeze(1)
    y_tr = torch.tensor(y_train_s, dtype=torch.float32)

    # Split off 10% validation
    n_val   = int(len(X_tr) * 0.1)
    n_train = len(X_tr) - n_val
    X_val, y_val = X_tr[n_train:], y_tr[n_train:]
    X_tr,  y_tr  = X_tr[:n_train],  y_tr[:n_train]

    train_loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=batch_size, shuffle=True
    )

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = CNN1D().to(device)
    opt    = torch.optim.Adam(model.parameters())
    loss_fn = nn.MSELoss()

    best_val, best_weights, wait = float('inf'), None, 0
    X_val, y_val = X_val.to(device), y_val.to(device)

    for epoch in range(epochs):
        # ── train ──
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

        # ── validate ──
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item()

        if val_loss < best_val:
            best_val = val_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break  # early stopping

    model.load_state_dict(best_weights)
    model.eval()
    
    # Predict in batches to save memory
    test_loader = DataLoader(TensorDataset(X_te), batch_size=4096, shuffle=False)
    y_pred_s_list = []
    with torch.no_grad():
        for xb, in test_loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy().flatten()
            y_pred_s_list.append(preds)
            
    y_pred_s = np.concatenate(y_pred_s_list).flatten()

    y_pred = sy.inverse_transform(y_pred_s.reshape(-1, 1)).flatten()
    rmse = np.sqrt(np.mean((y_test - y_pred)**2))
    return rmse, y_test, y_pred





In [ ]:
"""
step7_transformer.py — Transformer for Glucose Forecasting (Fixed)
==================================================================

Fixes from original:
1. Added Positional Encoding (crucial for time series in Transformers)
2. Stacked 3 encoder layers instead of 1
3. Added Cosine Annealing Learning Rate Scheduler
4. Removed the 50k sample cap (uses all data, ~200k+ windows)
"""

import numpy as np
import pandas as pd
import os
import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class PositionalEncoding(nn.Module):
    """
    Injects some information about the relative or absolute position of the
    tokens in the sequence. The positional encodings have the same dimension
    as the embeddings, so that the two can be summed.
    """
    def __init__(self, d_model, max_len=12):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        x shape: (batch_size, seq_len, d_model)
        """
        x = x + self.pe[:, :x.size(1), :]
        return x

class GlucoseTransformer(nn.Module):
    """
    Transformer encoder for glucose forecasting.

    Input:  (batch, seq_len, 1)  — 12 time-steps, 1 feature
    Output: (batch, 1)           — next glucose value
    """
    def __init__(self, seq_len=12, d_model=64, n_heads=4, num_layers=3, ff_dim=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_heads, 
            dim_feedforward=ff_dim, 
            dropout=dropout, 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        # x: (batch, seq_len, 1)
        x = self.input_proj(x)          # -> (batch, seq_len, d_model)
        x = self.pos_encoder(x)         # -> (batch, seq_len, d_model)
        x = self.transformer_encoder(x) # -> (batch, seq_len, d_model)
        
        # Use the representation of the LAST time step (or average pooling)
        # We'll use the last time step as it contains the most recent context
        x = x[:, -1, :]                 # -> (batch, d_model)
        
        return self.head(x).squeeze(-1) # -> (batch,)


def run_transformer(df, col_name, epochs=20, patience=3, batch_size=4096, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    X_train, X_test, y_train, y_test = prepare_column(df, col_name)

    # Scale to [0,1]
    sx = MinMaxScaler()
    sy = MinMaxScaler()
    # Flatten X to scale, then reshape
    orig_shape_tr = X_train.shape
    orig_shape_te = X_test.shape
    
    X_train_s = sx.fit_transform(X_train.reshape(-1, 1)).reshape(orig_shape_tr)
    X_test_s  = sx.transform(X_test.reshape(-1, 1)).reshape(orig_shape_te)
    y_train_s = sy.fit_transform(y_train.reshape(-1, 1)).flatten()

    # PyTorch tensors — shape (N, 12, 1)
    X_tr = torch.tensor(X_train_s, dtype=torch.float32).unsqueeze(-1)
    X_te = torch.tensor(X_test_s,  dtype=torch.float32).unsqueeze(-1)
    y_tr = torch.tensor(y_train_s, dtype=torch.float32)

    # Split off 10% validation
    n_val   = int(len(X_tr) * 0.1)
    n_train = len(X_tr) - n_val
    X_val, y_val = X_tr[n_train:], y_tr[n_train:]
    X_tr,  y_tr  = X_tr[:n_train],  y_tr[:n_train]

    train_loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=batch_size, shuffle=True
    )

    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model   = GlucoseTransformer().to(device)
    opt     = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()

    best_val, best_weights, wait = float('inf'), None, 0

    X_val = X_val.to(device)
    y_val = y_val.to(device)

    for epoch in range(epochs):
        # -- train --
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        
        scheduler.step()

        # -- validate --
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item()

        if val_loss < best_val:
            best_val = val_loss
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"    Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_weights)
    model.eval()
    
    # Predict in batches to save memory
    test_loader = DataLoader(TensorDataset(X_te), batch_size=4096, shuffle=False)
    y_pred_s_list = []
    with torch.no_grad():
        for xb, in test_loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy()
            y_pred_s_list.append(preds)
            
    y_pred_s = np.concatenate(y_pred_s_list).flatten()
    y_pred = sy.inverse_transform(y_pred_s.reshape(-1, 1)).flatten()
    rmse = np.sqrt(np.mean((y_test - y_pred)**2))
    
    return rmse, y_test, y_pred





In [ ]:
"""
step_statistics.py — Multi-run Statistical Significance Testing
===============================================================

Machine learning models (XGBoost, CNN, Transformer) have variance
due to random initialization and subsampling. A single run is not
enough to claim method A is better than method B.

This script:
1. Runs the prediction pipeline 5 times with different random seeds.
2. Collects RMSE across all runs.
3. Performs the Friedman test (non-parametric ANOVA) to check if 
   there are significant differences among methods.
4. If significant, performs post-hoc Nemenyi test to find exactly
   which methods are statistically different.
"""

import numpy as np
import pandas as pd
import os
from scipy.stats import friedmanchisquare
try:
    import scikit_posthocs as sp
except ImportError:
    sp = None

from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')


def run_multiple_seeds(df, cols, n_runs=5):
    """
    Run XGBoost, CNN, and Transformer multiple times with different random seeds.
    """
    results = []
    
    for run in range(n_runs):
        seed = 42 + run
        print(f"\n--- Run {run+1}/{n_runs} (Seed: {seed}) ---")
        
        for col in cols:
            print(f"  Evaluating {col}...")
            
            # XGBoost
            X_train, X_test, y_train, y_test = prepare_column(df, col)
            model = XGBRegressor(n_estimators=100, max_depth=6,
                                 learning_rate=0.05, random_state=seed, verbosity=0)
            model.fit(X_train, y_train, verbose=False)
            y_pred = model.predict(X_test)
            xgb_rmse = np.sqrt(np.mean((y_test - y_pred)**2))
            
            # Record
            results.append({
                'run': run+1,
                'seed': seed,
                'preprocessing': col,
                'model': 'XGBoost',
                'rmse': xgb_rmse
            })
            # CNN (only 3 runs)
            if run < 3:
                cnn_rmse, _, _ = run_cnn(df, col, seed=seed)
                results.append({
                    'run': run+1, 'seed': seed, 'preprocessing': col,
                    'model': 'CNN', 'rmse': cnn_rmse
                })
                
                tfm_rmse, _, _ = run_transformer(df, col, seed=seed)
                results.append({
                    'run': run+1, 'seed': seed, 'preprocessing': col,
                    'model': 'Transformer', 'rmse': tfm_rmse
                })

    return pd.DataFrame(results)


def statistical_analysis(results_df, save_path='results'):
    """
    Perform Friedman test and post-hoc Nemenyi test.
    """
    # 1. Summarize mean +/- std
    summary = results_df.groupby(['preprocessing', 'model'])['rmse'].agg(['mean', 'std']).reset_index()
    print("\n=== Mean +/- Std RMSE ===")
    print(summary.to_string(index=False))
    
    if sp is None:
        print("\n[!] scikit-posthocs not installed. Skipping Nemenyi test.")
        print("    pip install scikit-posthocs")
        return
        
    print("\n=== Statistical Tests ===")
    
    for model_name in ['XGBoost', 'CNN', 'Transformer']:
        print(f"\nModel: {model_name}")
        model_data = results_df[results_df['model'] == model_name]
        
        # Pivot table: rows=runs, cols=preprocessing, vals=rmse
        pivot = model_data.pivot(index='run', columns='preprocessing', values='rmse')
        
        # Friedman test
        stat, p = friedmanchisquare(*[pivot[c].values for c in pivot.columns])
        print(f"Friedman Test: Statistic={stat:.3f}, p-value={p:.3e}")
        
        if p < 0.05:
            print("  => Significant differences exist among preprocessing methods.")
            # Nemenyi post-hoc test
            nemenyi = sp.posthoc_nemenyi_friedman(pivot.values)
            nemenyi.columns = pivot.columns
            nemenyi.index = pivot.columns
            print("\nNemenyi p-values (p < 0.05 means significantly different):")
            print(nemenyi.round(3))
            nemenyi.to_csv(os.path.join(save_path, f'nemenyi_{model_name}.csv'))
        else:
            print("  => No significant differences detected.")


if __name__ == '__main__':
    os.makedirs('results', exist_ok=True)
    df = pd.read_csv('/kaggle/input/datasets/panditaadarsh/glucose-timeseries/hupa_smoothed.csv')
    
    cols = [c for c in df.columns if '_kalman' in c or '_smoothing_spline' in c or '_none' in c]
    
    print("Running Multi-Seed Statistical Analysis...")
    results_df = run_multiple_seeds(df, cols, n_runs=5)
    results_df.to_csv('results/multi_seed_results.csv', index=False)
    
    statistical_analysis(results_df)

